# Activity Cluster Analysis — Managerial Recommendations

**What this notebook does:**
1. Loads one or more OCEL `.jsonocel` logs.
2. Computes per-activity frequency, rework rate, delay, and stage-position metrics.
3. Clusters activities into behavioural groups using KMeans.
4. Maps each cluster to the 15-action manager action space.
5. For each case (or globally), prints an **if-then recommendation ladder** in plain managerial language.

---
**Input:** one or more `.jsonocel` files  
**Output:** printed recommendations + `cluster_report.csv` + `case_recommendations.csv`

## 0. Setup

In [1]:
%pip install pandas numpy scikit-learn pyarrow tabulate --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.1.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import warnings
from collections import defaultdict, Counter
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from tabulate import tabulate

warnings.filterwarnings('ignore')

# ── CONFIG ──────────────────────────────────────────────────────────────────
# List the .jsonocel files you want to analyse.
# You can add as many as you like; they will be pooled together.
LOG_PATHS = [
    './dataset/BPIC15_Municipality1.jsonocel',
    './dataset/BPIC15_Municipality2.jsonocel',
    './dataset/BPIC15_Municipality3.jsonocel',
    './dataset/BPIC15_Municipality4.jsonocel',
    './dataset/BPIC15_Municipality5.jsonocel',
]

OBJECT_TYPE   = 'Application'   # case identifier object type in the OCEL schema
N_CLUSTERS    = 6               # number of activity clusters (tuned below automatically)
OUTPUT_DIR    = Path('./output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Log paths configured:', len(LOG_PATHS))
print('Output dir:', OUTPUT_DIR.resolve())

Log paths configured: 5
Output dir: C:\Games\bureaucratic-workflow-analyzer\output


## 1. Load Logs

In [3]:
def load_ocel(path: str) -> dict:
    with open(path, 'r') as f:
        return json.load(f)

def extract_case_traces(log: dict, object_type: str = 'Application') -> dict:
    """Returns {case_id: [(timestamp, activity), ...]} sorted by timestamp."""
    events  = log['ocel:events']
    objects = log['ocel:objects']
    case_ids = {oid for oid, obj in objects.items() if obj['ocel:type'] == object_type}

    by_case = defaultdict(list)
    for ev_id, ev in events.items():
        ts  = pd.to_datetime(ev['ocel:timestamp'], utc=True, errors='coerce')
        act = ev['ocel:activity']
        for oid in ev.get('ocel:omap', []):
            if oid in case_ids:
                by_case[oid].append((ts, act))

    for case_id in by_case:
        by_case[case_id].sort(key=lambda x: x[0])

    return dict(by_case)

# ── Load all logs ────────────────────────────────────────────────────────────
all_traces = {}      # {source_tag: {case_id: [(ts, act)]}}
pooled_traces = {}   # flat {unique_case_id: [(ts, act)]}

for path in LOG_PATHS:
    p = Path(path)
    if not p.exists():
        print(f'  SKIP (not found): {p}')
        continue
    log    = load_ocel(path)
    traces = extract_case_traces(log, OBJECT_TYPE)
    tag    = p.stem
    all_traces[tag] = traces
    for cid, evs in traces.items():
        pooled_traces[f'{tag}::{cid}'] = evs
    print(f'  Loaded {tag}: {len(traces):,} cases')

print(f'\nTotal pooled cases: {len(pooled_traces):,}')

  Loaded BPIC15_Municipality1: 1,199 cases
  Loaded BPIC15_Municipality2: 832 cases
  Loaded BPIC15_Municipality3: 1,409 cases
  Loaded BPIC15_Municipality4: 1,053 cases
  Loaded BPIC15_Municipality5: 1,156 cases

Total pooled cases: 5,649


## 2. Compute Per-Activity Metrics

For every unique activity we compute five metrics that capture its behavioural profile:

| Metric | Meaning |
|--------|----------|
| `freq_global` | How many times it appears across all cases |
| `case_coverage` | Fraction of cases it appears in |
| `rework_rate` | Fraction of appearances that are repeats within the same case |
| `mean_stage_rank` | Normalised mean position in the trace (0 = early, 1 = late) |
| `mean_wait_hours` | Average waiting time before this activity executes |

In [4]:
from collections import defaultdict

act_freq      = Counter()          # total appearances
act_cases     = defaultdict(set)   # which cases contain this activity
act_rework    = Counter()          # repeat appearances (rework)
act_rank_sum  = defaultdict(float) # sum of normalised positions
act_rank_cnt  = Counter()          # count for rank averaging
act_wait_sum  = defaultdict(float) # sum of waiting hours
act_wait_cnt  = Counter()          # count for wait averaging

total_cases = len(pooled_traces)

for case_id, evs in pooled_traces.items():
    n = len(evs)
    seen_in_case = Counter()

    for i, (ts, act) in enumerate(evs):
        act_freq[act]      += 1
        act_cases[act].add(case_id)

        if seen_in_case[act] > 0:
            act_rework[act] += 1
        seen_in_case[act] += 1

        if n > 1:
            act_rank_sum[act] += i / (n - 1)
            act_rank_cnt[act] += 1

        if i > 0:
            prev_ts = evs[i - 1][0]
            wait_h  = max((ts - prev_ts).total_seconds() / 3600.0, 0.0)
            act_wait_sum[act] += wait_h
            act_wait_cnt[act] += 1

# ── Assemble metric table ────────────────────────────────────────────────────
records = []
for act in act_freq:
    freq     = act_freq[act]
    coverage = len(act_cases[act]) / total_cases
    rework   = act_rework[act] / freq if freq > 0 else 0.0
    rank     = act_rank_sum[act] / act_rank_cnt[act] if act_rank_cnt[act] > 0 else 0.5
    wait     = act_wait_sum[act] / act_wait_cnt[act] if act_wait_cnt[act] > 0 else 0.0
    records.append({
        'activity':       act,
        'freq_global':    freq,
        'case_coverage':  round(coverage, 4),
        'rework_rate':    round(rework, 4),
        'mean_stage_rank':round(rank, 4),
        'mean_wait_hours':round(wait, 4),
    })

metrics_df = pd.DataFrame(records).sort_values('freq_global', ascending=False).reset_index(drop=True)

print(f'Unique activities: {len(metrics_df)}')
print(f'\nTop 10 by frequency:')
print(tabulate(metrics_df.head(10), headers='keys', tablefmt='github', showindex=False))

Unique activities: 356

Top 10 by frequency:
| activity                                     |   freq_global |   case_coverage |   rework_rate |   mean_stage_rank |   mean_wait_hours |
|----------------------------------------------|---------------|-----------------|---------------|-------------------|-------------------|
| send confirmation receipt                    |          9090 |          0.9598 |        0.4035 |            0.1365 |           12.3128 |
| procedure change                             |          8649 |          0.84   |        0.4514 |            0.3832 |           42.702  |
| enter senddate decision environmental permit |          6943 |          0.7867 |        0.3599 |            0.8595 |           23.3954 |
| request complete                             |          6112 |          0.8977 |        0.1703 |            0.3863 |           70.9875 |
| phase application received                   |          5644 |          0.9989 |        0.0002 |            0.1469 |   

## 3. Cluster Activities

We use KMeans on the five standardised metrics. The optimal number of clusters is selected automatically using the silhouette score.

In [5]:
FEATURE_COLS = ['freq_global', 'case_coverage', 'rework_rate', 'mean_stage_rank', 'mean_wait_hours']

X_raw = metrics_df[FEATURE_COLS].values
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

# ── Auto-select best k via silhouette ────────────────────────────────────────
best_k, best_score, best_model = 2, -1, None
k_range = range(2, min(12, len(metrics_df) - 1))

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    score  = silhouette_score(X, labels)
    if score > best_score:
        best_k, best_score, best_model = k, score, km

print(f'Best k = {best_k}  |  Silhouette score = {best_score:.4f}')

metrics_df['cluster'] = best_model.labels_

# ── Cluster summary ──────────────────────────────────────────────────────────
cluster_summary = (
    metrics_df
    .groupby('cluster')[FEATURE_COLS]
    .agg(['mean', 'count'])
    .round(3)
)
cluster_summary.columns = ['_'.join(c) for c in cluster_summary.columns]
cluster_summary = cluster_summary.reset_index()

print('\nCluster profiles (means):')
summary_print = metrics_df.groupby('cluster')[FEATURE_COLS + ['activity']].agg(
    n_activities=('activity', 'count'),
    freq_mean=('freq_global', 'mean'),
    coverage_mean=('case_coverage', 'mean'),
    rework_mean=('rework_rate', 'mean'),
    stage_mean=('mean_stage_rank', 'mean'),
    wait_mean=('mean_wait_hours', 'mean'),
).round(3).reset_index()
print(tabulate(summary_print, headers='keys', tablefmt='github', showindex=False))

Best k = 4  |  Silhouette score = 0.6052

Cluster profiles (means):
|   cluster |   n_activities |   freq_mean |   coverage_mean |   rework_mean |   stage_mean |   wait_mean |
|-----------|----------------|-------------|-----------------|---------------|--------------|-------------|
|         0 |             16 |     654.562 |           0.068 |         0.421 |        0.591 |      42.15  |
|         1 |             39 |    4369.61  |           0.709 |         0.058 |        0.445 |      42.009 |
|         2 |            293 |     271.625 |           0.048 |         0.012 |        0.638 |      66.328 |
|         3 |              8 |     269.25  |           0.048 |         0.006 |        0.84  |    1588.28  |


## 4. Map Clusters → Action Space

Each cluster is characterised by its centroid profile and mapped to the most appropriate manager action from the 15-action space. The mapping is deterministic: it follows the same threshold logic used in Step 4 of the RL pipeline.

In [6]:
# ── Full action space (from Step 4) ─────────────────────────────────────────
ACTION_SPACE = {
    0:  'assign_to_primary_team',
    1:  'outsource_to_volunteer_pool',
    2:  'rebalance_overloaded_queue',
    3:  'merge_tasks_under_role',
    4:  'prioritize_urgent_case',
    5:  'defer_until_objections_resolved',
    6:  'escalate_to_higher_authority',
    7:  'skip_optional_subprocess',
    8:  'add_temporary_staff',
    9:  'adjust_staffing_by_case_volume',
    10: 'enable_cross_trained_pool',
    11: 'relax_rules_for_low_risk',
    12: 'trigger_high_cost_escalation',
    13: 'reroute_from_overloaded_employee',
    14: 'close_case',
}

# ── Managerial language for each action ─────────────────────────────────────
ACTION_JARGON = {
    0:  'Route to the primary processing team. No intervention required.',
    1:  'Capacity is strained. Consider outsourcing to an auxiliary or volunteer pool to absorb demand.',
    2:  'Queue imbalance detected. Redistribute workload across available staff to reduce local bottlenecks.',
    3:  'High-volume, low-complexity segment. Merge similar tasks under a single role to cut handover overhead.',
    4:  'Case age is elevated or a risk signal is active. Escalate case priority to prevent SLA breach.',
    5:  'Objection or appeal signal present. Hold further processing until outstanding objections are resolved.',
    6:  'Severe delay or suspension pathway detected. Escalate to senior authority for immediate decision.',
    7:  'Optional subprocess identified in a low-risk case. Safe to bypass; reduces cycle time.',
    8:  'Simultaneous volume spike and extreme delay. Authorise temporary staffing uplift.',
    9:  'Case intake volume exceeds baseline. Adjust planned staffing levels to match incoming load.',
    10: 'Workload or rework pressure is elevated. Activate cross-trained staff to cover multiple roles.',
    11: 'Low-risk, high-confidence pathway. Relax procedural checks to accelerate throughput.',
    12: 'High-risk case with prolonged delay. Trigger cost-escalation review before further expenditure.',
    13: 'Individual employee overload signal. Re-route cases from overloaded handler to available resource.',
    14: 'Terminal state reached. Formally close the case and archive.',
}

def map_cluster_to_action(row: pd.Series) -> int:
    """
    Deterministic if-then mapping from cluster centroid profile to action ID.
    Priority order: terminal → extreme risk → high delay → high rework →
    volume pressure → low risk → default.
    """
    freq     = row['freq_mean']
    coverage = row['coverage_mean']
    rework   = row['rework_mean']
    stage    = row['stage_mean']
    wait     = row['wait_mean']

    # Global percentile anchors (computed from the full metrics table)
    p75_freq    = metrics_df['freq_global'].quantile(0.75)
    p90_freq    = metrics_df['freq_global'].quantile(0.90)
    p75_wait    = metrics_df['mean_wait_hours'].quantile(0.75)
    p90_wait    = metrics_df['mean_wait_hours'].quantile(0.90)
    p75_rework  = metrics_df['rework_rate'].quantile(0.75)
    p90_rework  = metrics_df['rework_rate'].quantile(0.90)
    p25_cov     = metrics_df['case_coverage'].quantile(0.25)

    # ── If-then ladder ───────────────────────────────────────────────────────

    # 1. Terminal / final stage activities → close case
    if stage > 0.85 and coverage < p25_cov:
        return 14   # close_case

    # 2. Very high wait AND very high rework → trigger cost escalation review
    if wait >= p90_wait and rework >= p75_rework:
        return 12   # trigger_high_cost_escalation

    # 3. Extreme delay, mid-to-late stage → escalate to higher authority
    if wait >= p90_wait and stage >= 0.5:
        return 6    # escalate_to_higher_authority

    # 4. Very high rework, high frequency → rebalance the queue
    if rework >= p90_rework and freq >= p75_freq:
        return 2    # rebalance_overloaded_queue

    # 5. High rework + moderate delay → reroute from overloaded employee
    if rework >= p75_rework and wait >= p75_wait:
        return 13   # reroute_from_overloaded_employee

    # 6. Very high frequency + high coverage → volume management
    if freq >= p90_freq and coverage >= 0.6:
        # High volume AND delay → temporary staff
        if wait >= p75_wait:
            return 8    # add_temporary_staff
        else:
            return 9    # adjust_staffing_by_case_volume

    # 7. High frequency, wide coverage, low rework → cross-train or merge
    if freq >= p75_freq and coverage >= 0.5:
        if rework < 0.05:
            return 3    # merge_tasks_under_role
        else:
            return 10   # enable_cross_trained_pool

    # 8. High delay, early-to-mid stage → prioritize
    if wait >= p75_wait and stage < 0.6:
        return 4    # prioritize_urgent_case

    # 9. Low coverage, low rework, late stage → optional / low-risk
    if coverage < p25_cov and rework < 0.05 and stage > 0.6:
        return 7    # skip_optional_subprocess

    # 10. Low frequency, low rework, low wait → relax rules
    if freq < metrics_df['freq_global'].quantile(0.25) and rework < 0.1 and wait < p75_wait:
        return 11   # relax_rules_for_low_risk

    # 11. Default → assign to primary team
    return 0

# ── Apply mapping ────────────────────────────────────────────────────────────
cluster_action_map = {}
for _, row in summary_print.iterrows():
    cid    = int(row['cluster'])
    action = map_cluster_to_action(row)
    cluster_action_map[cid] = action

metrics_df['action_id']   = metrics_df['cluster'].map(cluster_action_map)
metrics_df['action_name'] = metrics_df['action_id'].map(ACTION_SPACE)
metrics_df['recommendation'] = metrics_df['action_id'].map(ACTION_JARGON)

print('\nCluster → Action mapping:')
for cid, aid in cluster_action_map.items():
    print(f'  Cluster {cid}  →  [{aid}] {ACTION_SPACE[aid]}')


Cluster → Action mapping:
  Cluster 0  →  [2] rebalance_overloaded_queue
  Cluster 1  →  [9] adjust_staffing_by_case_volume
  Cluster 2  →  [0] assign_to_primary_team
  Cluster 3  →  [6] escalate_to_higher_authority


## 5. Managerial Recommendation Ladder

For each cluster we compute the **probability that a randomly selected activity falls into it**, then produce a ranked recommendation ladder with plain-language managerial advice.

In [7]:
HEADER = '=' * 72
SEP    = '-' * 72

total_events = metrics_df['freq_global'].sum()

# Build cluster-level summary for the ladder
ladder_rows = []
for cid, aid in sorted(cluster_action_map.items()):
    subset = metrics_df[metrics_df['cluster'] == cid]
    cluster_events = subset['freq_global'].sum()
    prob = cluster_events / total_events if total_events > 0 else 0.0
    top_acts = subset.nlargest(3, 'freq_global')['activity'].tolist()
    ladder_rows.append({
        'cluster':      cid,
        'action_id':    aid,
        'action_name':  ACTION_SPACE[aid],
        'probability':  prob,
        'n_activities': len(subset),
        'top_activities': top_acts,
        'recommendation': ACTION_JARGON[aid],
    })

ladder_df = pd.DataFrame(ladder_rows).sort_values('probability', ascending=False).reset_index(drop=True)

# ── Print the ladder ─────────────────────────────────────────────────────────
print(HEADER)
print('  GLOBAL MANAGERIAL RECOMMENDATION LADDER')
print('  (ranked by probability of process activity falling in this cluster)')
print(HEADER)

for rank, row in enumerate(ladder_df.itertuples(), start=1):
    prob_pct = row.probability * 100
    print(f'\nRANK {rank}  |  Cluster {row.cluster}  |  P = {prob_pct:.1f}%')
    print(f'  Action   : [{row.action_id}] {row.action_name}')
    print(f'  Covers   : {row.n_activities} unique activities')
    print(f'  Examples : {" | ".join(row.top_activities[:3])}')
    print(f'  ► {row.recommendation}')
    print(SEP)

print('\nIf-then summary (copy-paste into report):')
print()
for rank, row in enumerate(ladder_df.itertuples(), start=1):
    prob_pct = row.probability * 100
    print(f'  IF   activity falls in Cluster {row.cluster} (prob ≈ {prob_pct:.1f}%)')
    print(f'  THEN {row.recommendation}')
    print()

  GLOBAL MANAGERIAL RECOMMENDATION LADDER
  (ranked by probability of process activity falling in this cluster)

RANK 1  |  Cluster 1  |  P = 64.9%
  Action   : [9] adjust_staffing_by_case_volume
  Covers   : 39 unique activities
  Examples : send confirmation receipt | procedure change | enter senddate decision environmental permit
  ► Case intake volume exceeds baseline. Adjust planned staffing levels to match incoming load.
------------------------------------------------------------------------

RANK 2  |  Cluster 2  |  P = 30.3%
  Action   : [0] assign_to_primary_team
  Covers   : 293 unique activities
  Examples : phase decision sent | record date of decision environmental permit | enter senddate decision
  ► Route to the primary processing team. No intervention required.
------------------------------------------------------------------------

RANK 3  |  Cluster 0  |  P = 4.0%
  Action   : [2] rebalance_overloaded_queue
  Covers   : 16 unique activities
  Examples : WAW permit a

## 6. Per-Case Recommendation

For each case we compute the distribution of its activities across clusters and generate a personalised recommendation based on the dominant cluster and secondary signals.

In [8]:
# Build activity → cluster lookup
act_to_cluster = dict(zip(metrics_df['activity'], metrics_df['cluster']))
act_to_action  = dict(zip(metrics_df['activity'], metrics_df['action_id']))

def case_recommendation(case_id: str, evs: list) -> dict:
    """
    Given a case trace (list of (timestamp, activity)), returns a
    recommendation dict with dominant action and probability breakdown.
    """
    n = len(evs)
    if n == 0:
        return None

    cluster_counts  = Counter()
    action_counts   = Counter()
    rework_count    = 0
    seen            = Counter()
    total_wait      = 0.0

    for i, (ts, act) in enumerate(evs):
        cid = act_to_cluster.get(act, -1)
        aid = act_to_action.get(act, 0)
        cluster_counts[cid]  += 1
        action_counts[aid]   += 1

        if seen[act] > 0:
            rework_count += 1
        seen[act] += 1

        if i > 0:
            total_wait += max((ts - evs[i-1][0]).total_seconds() / 3600.0, 0.0)

    case_duration_h = (evs[-1][0] - evs[0][0]).total_seconds() / 3600.0
    rework_rate     = rework_count / n
    mean_wait       = total_wait / max(n - 1, 1)

    # Cluster probability distribution
    cluster_probs = {k: v / n for k, v in cluster_counts.items()}
    dominant_cluster = cluster_counts.most_common(1)[0][0]
    dominant_action  = cluster_action_map.get(dominant_cluster, 0)

    # ── Override rules (if-then on case-level signals) ───────────────────────
    p75_wait   = metrics_df['mean_wait_hours'].quantile(0.75)
    p90_wait   = metrics_df['mean_wait_hours'].quantile(0.90)
    p75_rework = metrics_df['rework_rate'].quantile(0.75)

    final_action = dominant_action  # start with cluster signal

    if   rework_rate > 0.4 and mean_wait >= p90_wait:
        final_action = 12   # trigger_high_cost_escalation
    elif rework_rate > 0.4 and mean_wait >= p75_wait:
        final_action = 2    # rebalance_overloaded_queue
    elif mean_wait >= p90_wait and case_duration_h > 2000:
        final_action = 6    # escalate_to_higher_authority
    elif mean_wait >= p75_wait and rework_rate < 0.1:
        final_action = 4    # prioritize_urgent_case
    elif rework_rate > 0.3:
        final_action = 13   # reroute_from_overloaded_employee
    elif case_duration_h > 3000:
        final_action = 6    # escalate_to_higher_authority
    elif n <= 5 and rework_rate == 0.0:
        final_action = 11   # relax_rules_for_low_risk

    return {
        'case_id':          case_id,
        'n_events':         n,
        'case_duration_h':  round(case_duration_h, 2),
        'rework_rate':      round(rework_rate, 4),
        'mean_wait_h':      round(mean_wait, 4),
        'dominant_cluster': dominant_cluster,
        'cluster_probs':    cluster_probs,
        'final_action_id':  final_action,
        'final_action':     ACTION_SPACE[final_action],
        'recommendation':   ACTION_JARGON[final_action],
    }

# Run for all cases
case_recs = []
for case_id, evs in pooled_traces.items():
    rec = case_recommendation(case_id, evs)
    if rec:
        case_recs.append(rec)

case_recs_df = pd.DataFrame(case_recs)
print(f'Recommendations generated for {len(case_recs_df):,} cases.')
print('\nAction distribution across all cases:')
dist = case_recs_df['final_action'].value_counts()
for action, cnt in dist.items():
    pct = cnt / len(case_recs_df) * 100
    print(f'  {action:<40}  {cnt:5d}  ({pct:.1f}%)')

Recommendations generated for 5,649 cases.

Action distribution across all cases:
  adjust_staffing_by_case_volume             4043  (71.6%)
  escalate_to_higher_authority                820  (14.5%)
  prioritize_urgent_case                      562  (9.9%)
  assign_to_primary_team                      169  (3.0%)
  relax_rules_for_low_risk                     42  (0.7%)
  reroute_from_overloaded_employee             12  (0.2%)
  trigger_high_cost_escalation                  1  (0.0%)


## 7. Sample Case Recommendation Printout

Print the full recommendation ladder for a sample of individual cases.

In [9]:
N_SAMPLE = 5  # Change this to print more or fewer sample cases

# Pick a diverse sample: one per most common action
sample_cases = (
    case_recs_df
    .groupby('final_action', group_keys=False)
    .apply(lambda g: g.sample(1, random_state=42))
    .head(N_SAMPLE)
)

for _, rec in sample_cases.iterrows():
    print(HEADER)
    print(f'  CASE: {rec["case_id"]}')
    print(HEADER)
    print(f'  Events          : {rec["n_events"]}')
    print(f'  Case duration   : {rec["case_duration_h"]:.1f} hours  ({rec["case_duration_h"]/24:.1f} days)')
    print(f'  Rework rate     : {rec["rework_rate"]*100:.1f}%  (fraction of repeated activities)')
    print(f'  Mean wait time  : {rec["mean_wait_h"]:.2f} hours between steps')
    print(f'  Dominant cluster: Cluster {rec["dominant_cluster"]}'
          f'  → [{cluster_action_map.get(rec["dominant_cluster"],0)}]'
          f' {ACTION_SPACE.get(cluster_action_map.get(rec["dominant_cluster"],0),"")}')

    print(f'\n  Cluster probability breakdown:')
    for cid_str, prob in sorted(rec['cluster_probs'].items(), key=lambda x: -x[1]):
        cid   = int(cid_str) if not isinstance(cid_str, int) else cid_str
        aid   = cluster_action_map.get(cid, 0)
        aname = ACTION_SPACE[aid]
        bar   = '█' * int(prob * 30)
        print(f'    Cluster {cid:2d}  [{aid:2d}] {aname:<38}  {prob*100:5.1f}%  {bar}')

    print(f'\n  IF    the current state matches this case profile')
    print(f'  THEN  Action [{rec["final_action_id"]}]: {rec["final_action"]}')
    print(f'  ►     {rec["recommendation"]}')
    print()

  CASE: BPIC15_Municipality5::45588
  Events          : 50
  Case duration   : 1464.0 hours  (61.0 days)
  Rework rate     : 8.0%  (fraction of repeated activities)
  Mean wait time  : 29.88 hours between steps
  Dominant cluster: Cluster 1  → [9] adjust_staffing_by_case_volume

  Cluster probability breakdown:
    Cluster  1  [ 9] adjust_staffing_by_case_volume           64.0%  ███████████████████
    Cluster  2  [ 0] assign_to_primary_team                   34.0%  ██████████
    Cluster  0  [ 2] rebalance_overloaded_queue                2.0%  

  IF    the current state matches this case profile
  THEN  Action [9]: adjust_staffing_by_case_volume
  ►     Case intake volume exceeds baseline. Adjust planned staffing levels to match incoming load.

  CASE: BPIC15_Municipality5::44502
  Events          : 38
  Case duration   : 1450.8 hours  (60.5 days)
  Rework rate     : 7.9%  (fraction of repeated activities)
  Mean wait time  : 39.21 hours between steps
  Dominant cluster: Cluster 2  →

## 8. Save Outputs

In [10]:
# ── Cluster report ───────────────────────────────────────────────────────────
cluster_report = metrics_df[[
    'activity', 'freq_global', 'case_coverage', 'rework_rate',
    'mean_stage_rank', 'mean_wait_hours', 'cluster', 'action_id',
    'action_name', 'recommendation'
]].copy()

cluster_path = OUTPUT_DIR / 'cluster_report.csv'
cluster_report.to_csv(cluster_path, index=False)
print(f'Saved: {cluster_path}')

# ── Case recommendations ─────────────────────────────────────────────────────
case_out = case_recs_df[[
    'case_id', 'n_events', 'case_duration_h', 'rework_rate',
    'mean_wait_h', 'dominant_cluster', 'final_action_id',
    'final_action', 'recommendation'
]].copy()

cases_path = OUTPUT_DIR / 'case_recommendations.csv'
case_out.to_csv(cases_path, index=False)
print(f'Saved: {cases_path}')

# ── Global ladder ────────────────────────────────────────────────────────────
ladder_path = OUTPUT_DIR / 'global_ladder.csv'
ladder_df[['cluster', 'action_id', 'action_name', 'probability', 'n_activities', 'recommendation']].to_csv(ladder_path, index=False)
print(f'Saved: {ladder_path}')

print('\nDone. Three output files written to', OUTPUT_DIR.resolve())

Saved: output\cluster_report.csv
Saved: output\case_recommendations.csv
Saved: output\global_ladder.csv

Done. Three output files written to C:\Games\bureaucratic-workflow-analyzer\output
